# Posit Connect Inventory Scan

Discovers deployed content and stores its package inventory in PostgreSQL.

Configuration comes from environment variables set in this content item's **Vars** panel.

In [ ]:
from clients.connect_client import ConnectClient
from config.logging_config import configure_logging
from config.settings import get_settings
from database.connection import get_database
from services.inventory_service import InventoryService

configure_logging(level='INFO', log_dir=None, force=True)
settings = get_settings()
print('Connect  :', settings.connect_server_url)
print('Database :', settings.database_url_string())

## Connect and prepare the schema

In [ ]:
database = get_database(settings)
database.verify_connection()

# Alembic cannot run from a workstation with no route to the database,
# so the schema is created here on first run. Existing tables are untouched.
database.create_all()
database.verify_schema()
print('Schema ready.')

## Run the scan

In [ ]:
names = []


def on_discovered(count):
    print('Found', count, 'deployed applications.')
    print()


def on_progress(name, ok, count):
    names.append((name, ok, count))


with ConnectClient(settings) as client:
    info = client.verify_connection()
    print('Connected to Posit Connect', info['version'],
          'as', info['username'], '(role=' + str(info['user_role']) + ')')
    print()

    service = InventoryService(client, database, settings)
    result = service.run(on_discovered=on_discovered, progress=on_progress)

print()
print('Inventory Scan Complete')
print()
print('Applications Stored:', result.applications_stored)
print('Packages Stored:', result.packages_stored)

## Results

In [ ]:
for name, ok, count in names:
    mark = 'OK' if ok else '!!'
    print(mark, name, '-', count, 'packages')

if result.failures:
    print()
    print(len(result.failures), 'failure(s):')
    for guid, message in result.failures[:20]:
        print(' -', guid, ':', message[:150])

In [ ]:
from repositories.application_repository import ApplicationRepository
from repositories.package_repository import PackageRepository

with database.session() as session:
    apps = ApplicationRepository(session)
    packages = PackageRepository(session)
    print('Stored in PostgreSQL')
    print('  applications        :', apps.count())
    print('  packages            :', packages.count())
    print('  distinct owners     :', apps.distinct_owner_count())
    print('  unique pkg versions :', packages.distinct_package_versions())
    print('  by runtime          :', packages.package_type_breakdown())

database.dispose()